# Train.py – Training van het Moderatiemodel

Doel van het Script

Dit script is ontwikkeld om het moderatiemodel automatisch te trainen op zowel de oorspronkelijke dataset als op nieuwe afbeeldingen die tijdens gebruik van het systeem worden verzameld.

Het script combineert:

De originele UnsafeBench dataset
Door gebruikers beoordeelde afbeeldingen uit het geheugen van het systeem
Transfer learning met ResNet18
Data augmentation
Automatische opslag van het best presterende model

Hierdoor kan het model zichzelf periodiek verbeteren op basis van nieuwe voorbeelden.

Importeren van Bibliotheken
import os
import cv2
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
Wat gebeurt hier?

Alle benodigde bibliotheken worden geladen.

Deze bibliotheken worden gebruikt voor:

Bibliotheek	Functie
os	Bestandsbeheer
cv2	Beeldverwerking
numpy	Numerieke berekeningen
pandas	Datasetbeheer
PIL	Afbeeldingen verwerken
torch	Deep learning
torchvision	Pretrained modellen en transforms
Configuratie van het Trainingsproces
MEMORY_SAFE = "memory/safe"
MEMORY_UNSAFE = "memory/unsafe"

Deze mappen bevatten afbeeldingen die tijdens gebruik van het moderatiesysteem zijn verzameld.

Dataset Locatie
PICKLE_DATASET = "./dataset/proccesed_data.pkl"

Verwijst naar de eerder voorbewerkte dataset.

Model Opslag
MODEL_SAVE_PATH = (
    "model/resnet18_sketch_model_best.pth"
)

Hier wordt het best presterende model opgeslagen.

Epochs

Aantal keren dat de volledige dataset wordt doorlopen.

10 trainingsrondes
Learning Rate
0.0001

Bepaalt hoe snel het model leert.

Een lage waarde zorgt voor stabielere training.

GPU Ondersteuning
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
Wat gebeurt hier?

Er wordt gecontroleerd of een NVIDIA GPU beschikbaar is.

Wanneer CUDA aanwezig is:

GPU wordt gebruikt

Anders:

CPU wordt gebruikt

Dit maakt het script flexibel voor verschillende hardwareomgevingen.

Laden van de Oorspronkelijke Dataset
df = pd.read_pickle(
    PICKLE_DATASET
)

Hier wordt de eerder verwerkte UnsafeBench dataset geladen.

Features en Labels
X = list(
    df["sketch_image"].values
)

y = list(
    df["label"].values
)
X

Bevat de sketch-afbeeldingen.

y

Bevat de labels:

Waarde	Betekenis
0	Safe
1	Unsafe
Laden van Nieuwe Trainingsdata

Een belangrijk onderdeel van dit systeem is het gebruik van een geheugenmechanisme.

MEMORY_SAFE
MEMORY_UNSAFE

Deze mappen bevatten afbeeldingen die tijdens gebruik zijn verzameld.

Functie voor Laden van Geheugenafbeeldingen
def load_memory_images(folder, label):

Deze functie:

Doorloopt alle bestanden in een map
Leest iedere afbeelding in
Zet deze om naar grayscale
Koppelt een label
Geeft de afbeeldingen terug
Safe Afbeeldingen
safe_images, safe_labels =
    load_memory_images(
        MEMORY_SAFE,
        0
)

Alle afbeeldingen uit:

memory/safe

krijgen label:

0 = Safe
Unsafe Afbeeldingen
unsafe_images, unsafe_labels =
    load_memory_images(
        MEMORY_UNSAFE,
        1
)

Alle afbeeldingen uit:

memory/unsafe

krijgen label:

1 = Unsafe
Dataset Uitbreiden
X.extend(safe_images)
X.extend(unsafe_images)

y.extend(safe_labels)
y.extend(unsafe_labels)
Waarom is dit belangrijk?

Hier ontstaat een vorm van continue verbetering.

Nieuwe voorbeelden worden toegevoegd aan de oorspronkelijke trainingsdata.

Hierdoor leert het model van nieuwe situaties die niet in de oorspronkelijke dataset voorkwamen.

Dit lijkt op hoe veel moderne AI-systemen periodiek worden hertraind.

Data Augmentation

Voordat afbeeldingen naar het model gaan worden ze willekeurig aangepast.

transform = transforms.Compose(...)
Resize
Resize((224,224))

ResNet18 verwacht afbeeldingen van:

224 × 224 pixels
RGB Conversie
Grayscale(
    num_output_channels=3
)

Sketch-afbeeldingen bevatten slechts één kanaal.

ResNet18 verwacht drie kanalen.

Daarom wordt hetzelfde grijswaardenkanaal driemaal gebruikt.

Willekeurige Rotatie
RandomRotation(10)

Afbeeldingen kunnen maximaal 10 graden worden gedraaid.

Horizontal Flip
RandomHorizontalFlip()

Afbeeldingen worden soms gespiegeld.

Waarom Data Augmentation?

Het model ziet hierdoor telkens licht gewijzigde voorbeelden.

Voordelen:

Minder overfitting
Betere generalisatie
Robuuster model
Eigen Dataset Klasse
class SketchDataset(Dataset):

Deze klasse beheert:

afbeeldingen
labels
transformaties

en maakt de data compatibel met PyTorch.

Afbeeldingen Omzetten
if isinstance(image, np.ndarray):

    image = image.astype(
        np.uint8
    )

Zorgt ervoor dat OpenCV-afbeeldingen het juiste formaat hebben.

Trainings- en Validatieset
train_size = int(
    0.8 * len(dataset)
)

Verdeling:

Dataset	Percentage
Training	80%
Validation	20%
Waarom een Validatieset?

De validatieset wordt gebruikt om te controleren hoe goed het model presteert op onbekende data.

Dit geeft een realistischer beeld van de prestaties.

DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

De DataLoader verdeelt de data in batches.

Voordelen:

Efficiënter geheugenbeheer
Snellere training
Geschikt voor GPU-verwerking
Laden van ResNet18
model = models.resnet18(
    weights="DEFAULT"
)
Wat gebeurt hier?

Een vooraf getraind ResNet18-model wordt geladen.

Dit model bevat al kennis van miljoenen afbeeldingen uit de ImageNet-dataset.

Aanpassen van de Uitvoerlaag
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    2
)

De oorspronkelijke ImageNet-uitvoerlaag wordt vervangen.

Nieuwe klassen:

Safe
Unsafe
Lossfunctie en Optimizer
CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

Meet hoe fout de voorspellingen van het model zijn.

Adam Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

Past de gewichten van het netwerk aan tijdens training.

Trainingsproces

Voor iedere epoch wordt:

1. Een batch geladen
2. Een voorspelling gemaakt
3. De loss berekend
4. Backpropagation uitgevoerd
5. Het model geüpdatet

Train Accuracy
train_accuracy

Geeft aan hoeveel trainingsvoorbeelden correct werden geclassificeerd.

Validation Accuracy
val_accuracy

Meet prestaties op onbekende afbeeldingen.

Deze waarde is belangrijker dan de train accuracy.

Automatisch Opslaan van het Beste Model
if val_accuracy > best_accuracy:

Hier wordt gecontroleerd of het huidige model beter presteert dan eerdere versies.

Model Opslaan
torch.save(
    model.state_dict(),
    MODEL_SAVE_PATH
)

Alleen wanneer de validatie-accuracy verbetert wordt het model opgeslagen.

**Waarom is dit nuttig?**

Dit voorkomt dat een slechter model per ongeluk een beter model overschrijft.

Het systeem bewaart automatisch de beste versie die tijdens training is gevonden.

Dit principe staat bekend als:

**Model Checkpointing**

en wordt veel gebruikt binnen professionele machine learning pipelines.

Resultaat van het Script

Na afloop toont het script:

Training complete.
Best Validation Accuracy: XX.XX%
Model saved to:
model/resnet18_sketch_model_best.pth

Hiermee is direct zichtbaar:

Hoe goed het model presteerde
Welke versie is opgeslagen
Welke validatie-accuracy werd bereikt



# Conclusie

Dit trainingsscript vormt de kern van het moderatiesysteem. Het combineert de oorspronkelijke UnsafeBench-dataset met nieuwe afbeeldingen die tijdens gebruik van het systeem zijn verzameld. Vervolgens wordt een vooraf getraind ResNet18-model via transfer learning verder getraind op deze uitgebreide dataset. Door gebruik te maken van data augmentation, validatiecontroles en automatische modelopslag ontstaat een robuuste trainingspipeline die het model voortdurend kan verbeteren. Hierdoor blijft het moderatiesysteem zich aanpassen aan nieuwe voorbeelden van veilige en onveilige content, wat de betrouwbaarheid en toepasbaarheid van het systeem vergroot.